# Stage 2 — de-risk probe (fidelity 2a + headroom 2b)

Runs the ADR-0004 Stage-2 checks on **Colab** (the 18GB Mac keeps killing these multi-hour CFD runs). Uses the corrected 3D objective (`Blade3DObjective` / `evaluate_blade_aero_3d`, cycle-mean CFz, N_RADIAL=40).

- **2b headroom:** 8 designs varying only the rib wave -> does whole-fan wind vary enough to justify optimizing? (locally the first 3 already showed a ~1e12 spread: flat -3.1e11 -> gentle_dish +7.4e11 -> **yes**).
- **2a fidelity:** a 4-design subset also at fine fidelity -> does coarse-3D preserve the ranking (so the BO can use the cheap tier)?

Idempotent: checkpoints per run to Drive, safe to 'Run all' / resume after a drop.

## 1. Repo + deps + SU2 + Drive

In [ ]:
import importlib.util, os, subprocess, sys
from pathlib import Path
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(_v, "1")   # 1 thread/worker -> N processes on N cores

IN_COLAB = importlib.util.find_spec("google.colab") is not None
BRANCH = "main"  # Stage 1/2 objective rebuild is merged to main
REPO = Path("/content/fan-optimization") if IN_COLAB else Path.cwd()
if IN_COLAB:
    if not REPO.exists():
        subprocess.run(["git", "clone", "-b", BRANCH,
                        "https://github.com/clingergab/fan-optimization.git", str(REPO)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO), "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO), "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO), "pull", "origin", BRANCH], check=True)
    subprocess.run("apt-get install -qq -y libglu1-mesa libxrender1 libxcursor1 "
                   "libxft2 libxinerama1 unzip".split(), check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[bo]"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gmsh", "cadquery"], check=True)
    from google.colab import drive; drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/fanopt")
else:
    DRIVE_ROOT = REPO / "data"
for p in (str(REPO), str(REPO / "src"), str(REPO / "scripts")):
    if p not in sys.path:
        sys.path.insert(0, p)
print("repo:", REPO, "| drive:", DRIVE_ROOT)

In [ ]:
import urllib.request
from fanopt.cfd.phase3 import find_su2
SU2_BIN = find_su2()
if SU2_BIN is None and IN_COLAB:
    LOCAL = Path("/content/su2")
    if not any(LOCAL.rglob("SU2_CFD")):
        zc = DRIVE_ROOT / "su2" / "SU2-v8.0.1-linux64.zip"
        if not zc.exists():
            zc.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(
                "https://github.com/su2code/SU2/releases/download/v8.0.1/SU2-v8.0.1-linux64.zip", str(zc))
        LOCAL.mkdir(parents=True, exist_ok=True)
        subprocess.run(["unzip", "-q", "-o", str(zc), "-d", str(LOCAL)], check=True)
    hit = next(LOCAL.rglob("SU2_CFD"), None)
    if hit: subprocess.run(["chmod", "+x", str(hit)], check=False)
    SU2_BIN = str(hit) if hit else None
assert SU2_BIN, "SU2 not found"
print("SU2:", SU2_BIN)

## 2. Imports

In [ ]:
import json, math
from concurrent.futures import ProcessPoolExecutor, as_completed
from fanopt.bo.blade_objective_3d import whole_fan_j_fan
from fanopt.bo.redo_validation import headroom, ranking_agreement
from fanopt.cfd.blade_aero_3d import evaluate_blade_aero_3d
from fanopt.cfd.phase5 import VerifyConfig
from fanopt.geometry.blade import BladeParams
import fanopt.geometry.blade_cad as blade_cad
blade_cad.N_RADIAL_SECTIONS = 40  # the objective's geometry resolution (ADR-0004)
print("N_RADIAL_SECTIONS =", blade_cad.N_RADIAL_SECTIONS)

## 3. The design set (vary only the wave)

In [ ]:
# 8 designs that vary ONLY the rib wave (same thickness/panel/blade_count), so the spread of
# whole-fan cycle-mean CFz directly measures the wave's leverage on wind (2b). A subset also runs
# at FINE fidelity to check whether the cheap coarse tier preserves the ranking (2a).
COARSE = VerifyConfig(n_cycles=3, inner_iter=30)
FINE = VerifyConfig(n_cycles=5, inner_iter=60)
_BASE = dict(blade_count=8, rib_bow_interp="linear", t_rib_hub_m=0.004, t_rib_tip_m=0.004,
             panel_offsets_m=[[0.0, 0.0, 0.0]] * 4, panel_thickness_m=[[0.003] * 3] * 4)
WAVES = {
    "flat":        ([0.0005] * 5, "linear"),
    "gentle_dish": ([0.006, 0.012, 0.018, 0.022, 0.025], "linear"),
    "deep_dish":   ([0.010, 0.018, 0.026, 0.030, 0.030], "linear"),
    "zigzag":      ([0.025, 0.005, 0.028, 0.006, 0.026], "linear"),
    "smooth_wave": ([0.025, 0.005, 0.028, 0.006, 0.026], "smooth"),
    "tip_heavy":   ([0.002, 0.006, 0.012, 0.022, 0.030], "linear"),
    "hub_heavy":   ([0.030, 0.022, 0.012, 0.006, 0.002], "linear"),
    "mid_bump":    ([0.005, 0.020, 0.030, 0.020, 0.005], "linear"),
}
FIDELITY_SUBSET = ("flat", "gentle_dish", "deep_dish", "zigzag")  # run at FINE too (2a)
def _design(name):
    knots, interp = WAVES[name]
    return BladeParams.from_dict({**_BASE, "rib_bow_knots_m": knots, "rib_bow_interp": interp})
print(len(WAVES), "designs;", len(FIDELITY_SUBSET), "also at fine fidelity")

## 4. Run (coarse all + fine subset) — resumable

In [ ]:
OUT = DRIVE_ROOT / "stage2_probe"; OUT.mkdir(parents=True, exist_ok=True)
_rf = OUT / "results.json"
N_WORKERS = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else (os.cpu_count() or 1)

def _run(job):
    name, tier = job
    cfg = COARSE if tier == "coarse" else FINE
    p = _design(name)
    try:
        res = evaluate_blade_aero_3d(p, OUT / f"{name}_{tier}", cfg=cfg, su2_bin=SU2_BIN)
        return {"name": name, "tier": tier, "j_fan_whole": whole_fan_j_fan(res.j_fan_mean, p.blade_count)}
    except Exception as e:
        return {"name": name, "tier": tier, "error": f"{type(e).__name__}: {e}"}

jobs = [(n, "coarse") for n in WAVES] + [(n, "fine") for n in FIDELITY_SUBSET]
done = json.loads(_rf.read_text()) if _rf.exists() else []
have = {(d["name"], d["tier"]) for d in done}
todo = [j for j in jobs if j not in have]  # idempotent / resumable
print(f"{len(have)} cached, {len(todo)} to run on {N_WORKERS} workers")
if todo:
    with ProcessPoolExecutor(max_workers=N_WORKERS) as pool:
        futs = {pool.submit(_run, j): j for j in todo}
        for fut in as_completed(futs):
            r = fut.result(); done.append(r); _rf.write_text(json.dumps(done, indent=2))
            tag = r.get("error") or f"{r['j_fan_whole']:+.3e}"
            print(f"  {r['name']}/{r['tier']}: {tag}", flush=True)
print("done:", len(done), "runs")

## 5. Conclusions — 2b headroom + 2a fidelity

In [ ]:
res = json.loads((DRIVE_ROOT / "stage2_probe" / "results.json").read_text())
coarse = {r["name"]: r["j_fan_whole"] for r in res if r["tier"] == "coarse" and "error" not in r}
fine = {r["name"]: r["j_fan_whole"] for r in res if r["tier"] == "fine" and "error" not in r}

print("=== 2b HEADROOM: does 3D wind vary enough with the wave to optimize? ===")
for n in sorted(coarse, key=lambda n: -coarse[n]):
    print(f"  {n:12s} whole-fan CFz={coarse[n]:+.3e}")
h = headroom(list(coarse.values()))
print(f"  -> n={h.n} range_frac={h.range_frac:.2f} HEADROOM={h.has_headroom}\n")

print("=== 2a FIDELITY: does coarse-3D preserve the fine-3D ranking? ===")
names = [n for n in FIDELITY_SUBSET if n in coarse and n in fine]
for n in names:
    print(f"  {n:12s} coarse={coarse[n]:+.3e}  fine={fine[n]:+.3e}")
a = ranking_agreement([coarse[n] for n in names], [fine[n] for n in names])
print(f"  -> n={a.n} kendall_tau={a.kendall_tau} PRESERVED={a.ranking_preserved}")